In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *

In [ ]:
import numpy as np
import pandas as pd

# --- Black-Box Graph Generator ---

# Randomize total nodes (close to 10,000, but not fixed)
n = np.random.randint(9500, 10501)

# Randomly choose the community balance (fraction for community 0 between 0.2 and 0.8)
balance = np.random.uniform(0.3, 0.7)
n_comm0 = int(np.round(balance * n))
n_comm1 = n - n_comm0

# Create opaque community assignments
Z = np.zeros(n, dtype=int)
indices = np.arange(n)
np.random.shuffle(indices)
Z[indices[:n_comm0]] = 0
Z[indices[n_comm0:]] = 1

# Internally generate hidden "attributes" for each node that drive edge formation.
# (The specifics remain undisclosed, but they are engineered to produce communities with similar behaviors.)
hidden_factors = np.random.randn(n, 3)  # extra dimensions to hide clarity

# Apply a non-trivial transformation on the hidden factors that creates subtle differences.
transformed = np.tanh(hidden_factors + np.random.uniform(-0.5, 0.5, size=hidden_factors.shape))

# --- Create edges with a black-box probabilistic model ---
edges = []
num_candidates = round(n / 100)  # each node considers a small, random subset of all other nodes

for i in range(n):
    # Randomly choose candidate neighbors (excluding itself)
    candidates = np.random.choice(np.delete(np.arange(n), i), size=num_candidates, replace=False)
    for j in candidates:
        # Compute a hidden measure that determines edge likelihood.
        metric = np.abs(np.sum(transformed[i] - transformed[j])) / 3
        # Internally adjust probabilities, making intra- and inter-community behavior hard to separate.
        if Z[i] == Z[j]:
            prob = np.exp(-metric) * np.random.uniform(0.8, 1.2)
        else:
            prob = np.exp(-metric) * np.random.uniform(0.8, 1.2)
        # A slight bias to allow a moderate number of edges overall.
        if np.random.rand() < prob * 0.4:
            weight = np.random.rand()
            # Ensure a consistent ordering of nodes to avoid duplicate edges.
            edge = tuple(sorted((i, j)))
            edges.append((edge[0], edge[1], weight))

# Remove potential duplicate edges and prepare the DataFrame
unique_edges = list(set(edges))
df_edges = pd.DataFrame(unique_edges, columns=["node1", "node2", "weight"])

# Save results to CSV files
#df_edges.to_csv("synthetic_graph_edges.csv", index=False)
#df_communities = pd.DataFrame({"node": np.arange(n), "community": Z})
#df_communities.to_csv("synthetic_graph_communities.csv", index=False)

print("Black-box synthetic graph generated:")
print(f" - Total nodes: {n}")
print(f" - Community 0: {n_comm0}, Community 1: {n_comm1}")
print(f" - Total edges: {len(unique_edges)}")
print("Output files: 'synthetic_graph_edges.csv' and 'synthetic_graph_communities.csv'")


Black-box synthetic graph generated:
 - Total nodes: 10175
 - Community 0: 6190, Community 1: 3985
 - Total edges: 284799
Output files: 'synthetic_graph_edges.csv' and 'synthetic_graph_communities.csv'


In [4]:
for emb_mode, p22 in product(EMB_MODES[:1], P22S):
	print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
	metrics = {}
	for rho, pi in product(RHOS, PIS):
		metrics[(rho, pi)] = {}
		for model, model_params in MODELS_AND_PARAMS:
			m = model(rho, pi, model_params, p22 = p22)
			A, Z = m(42)
			metrics[(rho, pi)][m] = {}
			#for t in TRANSFORMS_EXT:
			for t in TRANSFORMS_THR_QTL:
				print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
				metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

	plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
	for rho, pi in product(RHOS, PIS):
		#subfolder = f"Tranforms_Beta_Lognormal"
		subfolder = f"Threshold_Quantile_Beta_Lognormal"
		plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=Beta, transformation=Quantile

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.5, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.5, model=B

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Threshold (τ = 0.05)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Threshold (τ = 0.1)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Quantile (q = 0.1)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Quantile (q = 0.25)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Quantile (q = 0.5)
Simulating for rho=0.5, pi=0.5, model=Beta, transformation=Threshold (τ = 0.01)
Simulating for rho=0.5, pi=0.5, model=Beta, transfor

In [5]:
for emb_mode, p22 in product(EMB_MODES[:1], P22S):
	print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
	metrics = {}
	for rho, pi in product(RHOS, PIS):
		metrics[(rho, pi)] = {}
		for model, model_params in product([lognormWSBM], [(1, 1), (0.5, 1), (1, 0.5), (0.1, 0.5)]):
			m = model(rho, pi, model_params, p22 = p22)
			A, Z = m(42)
			metrics[(rho, pi)][m] = {}
			#for t in TRANSFORMS_EXT:
			for t in TRANSFORMS_THR_QTL:
				print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
				metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

	plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
	for rho, pi in product(RHOS, PIS):
		#subfolder = f"Tranforms_Lognormal"
		subfolder = f"Threshold_Quantile_Lognormal"
		plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.25, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.25, pi=0.5, model=L

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (2). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.5, pi=0.1, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.1)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.1)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.25)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Quantile (q = 0.5)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.01)
Simulating for rho=0.5, pi=0.5, model=LogN, transformation=Threshold (τ = 0.05)
Simulating for rho=0.5, pi=0.5, model=LogN, transfor

In [ ]:
path = "Computation/Grids"
n_batch = 1
metrics_g = {}
metrics_g_1st_layer = {}
for rho, pi, model in RHOS_PIS_MODELS:
	file = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	grids_stacked = [np.load(f"{file}_{b}.npz") for b in range(n_batch)]
	metrics_g[(rho, pi, model)] = {}
	metrics_g_1st_layer[(rho, pi, model)] = {}
	for t in TRANSFORMS:
		metrics_g[(rho, pi, model)][t] = {}
		metrics_g[(rho, pi, model)][t]['std'] = {}
		metrics_g_1st_layer[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			#grid_1st_layer = grids_stacked[f'{t.id}_{metric}'][:, :, 0]
			g_stack = np.concatenate([g[f'{t.id}_{metric}'] for g in grids_stacked], axis = -1)
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std
			metrics_g_1st_layer[(rho, pi, model)][t][metric] = g_stack[:, :, 0]

metrics_g = aggregate_metrics(metrics_g)
metrics_g_1st_layer = aggregate_metrics(metrics_g_1st_layer)

metrics_g = best_transform_metrics(metrics_g)

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Pro

In [ ]:
plot_scatter_Rand_vs_Chernoff(metrics_g_1st_layer, n_points_ratio_displayed=0.2)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in [TRANSFORMS[2]]:
		m = metrics_g[(rho, pi, model)][t]
		plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)
		#art_plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

model = Beta, rho = 0.25, pi = 0.1
m_id = C_true
vmin = 1.5911911657059483e-06, vmax = 0.06186009598735778
0.015319485734570528 [1.59119117e-06 1.77156179e-06 2.31996664e-06 ... 6.18600660e-02
 6.18600811e-02 6.18600960e-02]
model = Beta, rho = 0.25, pi = 0.1
m_id = C_graph
vmin = 8.452864739346986e-05, vmax = 0.11464264208607078
0.017213034230368747 [8.45286474e-05 1.11022737e-04 1.15801262e-04 ... 1.13085656e-01
 1.13439180e-01 1.14642642e-01]
model = Beta, rho = 0.25, pi = 0.1
m_id = C_embed
vmin = 3.0040065050805556e-05, vmax = 0.12788913568116095
0.013793399615013578 [3.00400651e-05 3.77838843e-05 3.93383002e-05 ... 1.25966010e-01
 1.26407916e-01 1.27889136e-01]
model = LogN, rho = 0.25, pi = 0.1
m_id = C_true
vmin = 0.0, vmax = 0.0
0.0 [0.]


ValueError: Invalid vmin or vmax

Error in callback <function _draw_all_if_interactive at 0x000002B971C1F3A0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Invalid vmin or vmax

ValueError: Invalid vmin or vmax

<Figure size 1150x1000 with 6 Axes>

In [ ]:
# Prendre moins de place première ligne

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plot_bias_heatmap(rho, pi, model, t, m, log = True)
		#art_plot_bias_heatmap(rho, pi, model, t, m, log = True)

In [ ]:
# Rand moyen, Regret moyen pour Best transform overall
# Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area C-estim-Best Transform) over 8 graphs (TreeMap)
# Pour chaque Transform élue Best Transform
#	  - Rand moyen + Regret moyen
#     Puis découpage en 2 régions (Regret > 0 et Regret = 0) et Rand moyen et Regret moyen

# Average(Area Rand-Best Transform) over 8 graphs (TreeMap)
#     Rand moyen pour chaque Best transform

# Average(Rand) over 8 graphs for 4 transforms + Best

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plot_best_transform_heatmaps(rho, pi, model, m)
	#art_plot_best_transform_heatmaps(rho, pi, model, m)

In [7]:
emb_mode = 'sqrt-scaled'
p22 = 'fixed'
fixed_param = '11'

plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

In [ ]:
path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}/Lines"

metrics_l = {}
for p in linspace_exclusive(0, 1, 4):
	metrics_l[p] = {}
	for rho, pi, model in RHOS_PIS_MODELS:
		param_str = f'{model.param_name}{fixed_param}'
		file = f"{model.__name__}_{rho}_{pi}_{param_str}_{p}".replace('.', '')
		lines_stacked = np.load(f"{path}/{file}.npz")
		metrics_l[p][(rho, pi, model)] = {}
		metrics_l[p][(rho, pi, model)]['fixed_param'] = lines_stacked['fixed_param']
		for tid, t in TRANSFORMS_MAP.items():
			metrics_l[p][(rho, pi, model)][t] = {}
			metrics_l[p][(rho, pi, model)][t]['std'] = {}
			for metric in METRICS_ID:
				#line = local_weighted_average(lines_stacked[f'{tid}_{metric}'])
				mean = np.mean(lines_stacked[f'{tid}_{metric}'], axis = -1)
				std = np.std(lines_stacked[f'{tid}_{metric}'], axis = -1)
				metrics_l[p][(rho, pi, model)][t][metric] = mean
				metrics_l[p][(rho, pi, model)][t]['std'][metric] = std
			
for p in linspace_exclusive(0, 1, 4):
	for rho, pi, model in RHOS_PIS_MODELS:
		m = metrics_l[p][(rho, pi, model)]
		metrics_l[p][(rho, pi, model)] = best_transform_metrics(m)

In [9]:
for p in linspace_exclusive(0, 1, 4):
	for rho, pi, model in RHOS_PIS_MODELS:
		m_l = metrics_l[p][(rho, pi, model)]
		plotter.plot_best_transform_lines_light(rho, pi, model, m_l)